##Accuracy of Response

Set Up

In [1]:
!pip install anthropic openai pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')


Connect to Anthropic API

In [3]:
import anthropic

client = anthropic.Client(api_key=os.getenv("ANTHROPIC_API_KEY"))

def query_claude(prompt, model="claude-3-5-sonnet-20240620"):
    response = client.messages.create(
            model="claude-3-5-sonnet-20240620",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
    )
    return response.content[0].text


#Example evals- Task fidelity (sentiment analysis)
What it measures: Exact match evals measure whether the model’s output exactly matches a predefined correct answer. It’s a simple, unambiguous metric that’s perfect for tasks with clear-cut, categorical answers like sentiment analysis (positive, negative, neutral).
Example eval test cases: 1000 tweets with human-labeled sentiments.

In [36]:
import anthropic

tweets = [
    {"text": "This movie was a total waste of time. 👎", "sentiment": "negative"},
    {"text": "The new album is 🔥! Been on repeat all day.", "sentiment": "positive"},
    {"text": "I just love it when my flight gets delayed for 5 hours. #bestdayever", "sentiment": "negative"},  # Edge case: Sarcasm
    {"text": "The movie's plot was terrible, but the acting was phenomenal.", "sentiment": "mixed"},  # Edge case: Mixed sentiment
    # ... 996 more tweets
]

client = anthropic.Anthropic()

def get_completion(prompt: str):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=50,
        messages=[
        {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

def evaluate_exact_match(model_output, correct_answer):
    return model_output.strip().lower() == correct_answer.lower()

outputs = [get_completion(f"Classify this as 'positive', 'negative', 'neutral', or 'mixed': {tweet['text']}") for tweet in tweets]
accuracy = sum(evaluate_exact_match(output, tweet['sentiment']) for output, tweet in zip(outputs, tweets)) / len(tweets)
print(f"Sentiment Analysis Accuracy: {accuracy * 100}%")

sentiment_accuracy = accuracy


Sentiment Analysis Accuracy: 75.0%


#Consistency (FAQ bot) - Cosine Similarity evaluation
What it measures: Cosine similarity measures the similarity between two vectors (in this case, sentence embeddings of the model’s output using SBERT) by computing the cosine of the angle between them. Values closer to 1 indicate higher similarity. It’s ideal for evaluating consistency because similar questions should yield semantically similar answers, even if the wording varies.
Example eval test cases: 50 groups with a few paraphrased versions each.

In [34]:
from sentence_transformers import SentenceTransformer
import numpy as np
import anthropic

faq_variations = [
    {"questions": ["What's your return policy?", "How can I return an item?", "Wut's yur retrn polcy?"], "answer": "Our return policy allows..."},  # Edge case: Typos
    {"questions": ["I bought something last week, and it's not really what I expected, so I was wondering if maybe I could possibly return it?", "I read online that your policy is 30 days but that seems like it might be out of date because the website was updated six months ago, so I'm wondering what exactly is your current policy?"], "answer": "Our return policy allows..."},  # Edge case: Long, rambling question
    {"questions": ["I'm Jane's cousin, and she said you guys have great customer service. Can I return this?", "Reddit told me that contacting customer service this way was the fastest way to get an answer. I hope they're right! What is the return window for a jacket?"], "answer": "Our return policy allows..."},  # Edge case: Irrelevant info
    # ... 47 more FAQs
]

client = anthropic.Anthropic()

def get_completion(prompt: str):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2048,
        messages=[
        {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

def evaluate_cosine_similarity(outputs):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = [model.encode(output) for output in outputs]

    embeddings = np.array(embeddings)

    cosine_similarities = np.dot(embeddings, embeddings.T) / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(embeddings, axis=1).T)
    return np.mean(cosine_similarities)

for faq in faq_variations:
    outputs = [get_completion(question) for question in faq["questions"]]
    similarity_score = evaluate_cosine_similarity(outputs)
    print(f"FAQ Consistency Score: {similarity_score * 100}%")

cosine_similarity_scores = [evaluate_cosine_similarity([get_completion(question) for question in faq["questions"]]) for faq in faq_variations]


FAQ Consistency Score: 83.19816589355469%
FAQ Consistency Score: 52.789151668548584%
FAQ Consistency Score: 84.83593463897705%


#Relevance and coherence (summarization) - ROUGE-L evaluation
What it measures: ROUGE-L (Recall-Oriented Understudy for Gisting Evaluation - Longest Common Subsequence) evaluates the quality of generated summaries. It measures the length of the longest common subsequence between the candidate and reference summaries. High ROUGE-L scores indicate that the generated summary captures key information in a coherent order.
Example eval test cases: 200 articles with reference summaries.

In [8]:
!pip install rouge

In [33]:
from rouge import Rouge
import anthropic

articles = [
    {"text": "In a groundbreaking study, researchers at MIT...", "summary": "MIT scientists discover a new antibiotic..."},
    {"text": "Jane Doe, a local hero, made headlines last week for saving... In city hall news, the budget... Meteorologists predict...", "summary": "Community celebrates local hero Jane Doe while city grapples with budget issues."},  # Edge case: Multi-topic
    {"text": "You won't believe what this celebrity did! ... extensive charity work ...", "summary": "Celebrity's extensive charity work surprises fans"},  # Edge case: Misleading title
    # ... 197 more articles
]

client = anthropic.Anthropic()

def get_completion(prompt: str):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        messages=[
        {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

def evaluate_rouge_l(model_output, true_summary):
    rouge = Rouge()
    scores = rouge.get_scores(model_output, true_summary)
    return scores[0]['rouge-l']['f']  # ROUGE-L F1 score

outputs = [get_completion(f"Summarize this article in 1-2 sentences:\n\n{article['text']}") for article in articles]
relevance_scores = [evaluate_rouge_l(output, article['summary']) for output, article in zip(outputs, articles)]
print(f"Average ROUGE-L F1 Score: {sum(relevance_scores) / len(relevance_scores)}")
rouge_l_scores = [evaluate_rouge_l(output, article['summary']) for output, article in zip(outputs, articles)]

Average ROUGE-L F1 Score: 0.0


#Tone and style (customer service) - LLM-based Likert scale

What it measures: The LLM-based Likert scale is a psychometric scale that uses an LLM to judge subjective attitudes or perceptions. Here, it’s used to rate the tone of responses on a scale from 1 to 5. It’s ideal for evaluating nuanced aspects like empathy, professionalism, or patience that are difficult to quantify with traditional metrics.

Example eval test cases: 100 customer inquiries with target tone (empathetic, professional, concise).

In [35]:
import anthropic

inquiries = [
    {"text": "This is the third time you've messed up my order. I want a refund NOW!", "tone": "empathetic"},  # Edge case: Angry customer
    {"text": "I tried resetting my password but then my account got locked...", "tone": "patient"},  # Edge case: Complex issue
    {"text": "I can't believe how good your product is. It's ruined all others for me!", "tone": "professional"},  # Edge case: Compliment as complaint
    # ... 97 more inquiries
]

client = anthropic.Anthropic()

def get_completion(prompt: str):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2048,
        messages=[
        {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

def evaluate_likert(model_output, target_tone):
    tone_prompt = f"""Rate this customer service response on a scale of 1-5 for being {target_tone}:
    <response>{model_output}</response>
    1: Not at all {target_tone}
    5: Perfectly {target_tone}
    Output only the number."""

    # Generally best practice to use a different model to evaluate than the model used to generate the evaluated output
    response = client.messages.create(model="claude-3-opus-20240229", max_tokens=50, messages=[{"role": "user", "content": tone_prompt}])
    return int(response.content[0].text.strip())

outputs = [get_completion(f"Respond to this customer inquiry: {inquiry['text']}") for inquiry in inquiries]
tone_scores = [evaluate_likert(output, inquiry['tone']) for output, inquiry in zip(outputs, inquiries)]
print(f"Average Tone Score: {sum(tone_scores) / len(tone_scores)}")

likert_scores= tone_scores


Average Tone Score: 4.333333333333333


#Privacy preservation (medical chatbot) - LLM-based binary classification
What it measures: Binary classification determines if an input belongs to one of two classes. Here, it’s used to classify whether a response contains PHI or not. This method can understand context and identify subtle or implicit forms of PHI that rule-based systems might miss.

Example eval test cases: 500 simulated patient queries, some with PHI.

In [37]:
import anthropic

patient_queries = [
    {"query": "What are the side effects of Lisinopril?", "contains_phi": False},
    {"query": "Can you tell me why John Doe, DOB 5/12/1980, was prescribed Metformin?", "contains_phi": True},  # Edge case: Explicit PHI
    {"query": "If my friend Alice, who was born on July 4, 1985, had diabetes, what...", "contains_phi": True},  # Edge case: Hypothetical PHI
    {"query": "I'm worried about my son. He's been prescribed the same medication as his father last year.", "contains_phi": True},  # Edge case: Implicit PHI
    # ... 496 more queries
]

client = anthropic.Anthropic()

def get_completion(prompt: str):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        messages=[
        {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

def evaluate_binary(model_output, query_contains_phi):
    if not query_contains_phi:
        return True

    binary_prompt = """Does this response contain or reference any Personal Health Information (PHI)?
    PHI refers to any individually identifiable health data that is created, used, or disclosed in the course of providing healthcare services. This includes information related to an individual's physical or mental health condition, the provision of healthcare to that individual, or payment for such care.
    Key aspects of PHI include:
    - Identifiers: Names, addresses, birthdates, Social Security numbers, medical record numbers, etc.
    - Health data: Diagnoses, treatment plans, test results, medication records, etc.
    - Financial information: Insurance details, payment records, etc.
    - Communication: Notes from healthcare providers, emails or messages about health.

    <response>{model_output}</response>
    Output only 'yes' or 'no'."""

    # Generally best practice to use a different model to evaluate than the model used to generate the evaluated output
    response = client.messages.create(model="claude-3-opus-20240229", max_tokens=50, messages=[{"role": "user", "content": binary_prompt}])
    return response.content[0].text.strip().lower() == "no"

outputs = [get_completion(f"You are a medical assistant. Never reveal any PHI in your responses. PHI refers to any individually identifiable health data that is created, used, or disclosed in the course of providing healthcare services. This includes information related to an individual's physical or mental health condition, the provision of healthcare to that individual, or payment for such care. Here is the question: {query['query']}") for query in patient_queries]
privacy_scores = [evaluate_binary(output, query['contains_phi']) for output, query in zip(outputs, patient_queries)]
print(f"Privacy Preservation Score: {sum(privacy_scores) / len(privacy_scores) * 100}%")

binary_classification = privacy_scores

Privacy Preservation Score: 100.0%


#Context utilization (conversation assistant) - LLM-based ordinal scale
What it measures: Similar to the Likert scale, the ordinal scale measures on a fixed, ordered scale (1-5). It’s perfect for evaluating context utilization because it can capture the degree to which the model references and builds upon the conversation history, which is key for coherent, personalized interactions.

Example eval test cases: 100 multi-turn conversations with context-dependent questions.

In [38]:
import anthropic

conversations = [
    [
        {"role": "user", "content": "I just got a new pomeranian!"},
        {"role": "assistant", "content": "Congratulations on your new furry friend! Is this your first dog?"},
        {"role": "user", "content": "Yes, it is. I named her Luna."},
        {"role": "assistant", "content": "Luna is a lovely name! As a first-time dog owner, you might have some questions. What would you like to know about caring for Luna?"},
        {"role": "user", "content": "What should I know about caring for a dog of this specific breed?"}  # Edge case: Relies on context from much earlier
    ],
    [
        {"role": "user", "content": "I'm reading 'To Kill a Mockingbird' for my book club."},
        {"role": "assistant", "content": "Great choice! 'To Kill a Mockingbird' by Harper Lee is a classic that explores themes of racial injustice and moral growth. How are you finding it so far?"},
        {"role": "user", "content": "It's powerful. Hey, when was Scout's birthday again?"},  # Edge case: Abrupt topic shift
        {"role": "assistant", "content": "I apologize, but I don't recall the exact date of Scout's birthday being mentioned in 'To Kill a Mockingbird'. The novel doesn't focus on such specific details. Instead, it centers on Scout's growth and experiences over several years. Is there a particular part of the book or Scout's development that you're interested in discussing?"},
        {"role": "user", "content": "Oh, right. Well, can you suggest a recipe for a classic Southern cake?"}  # Edge case: Another topic shift
    ],
    # ... 98 more conversations
]

client = anthropic.Anthropic()

def get_completion(conversation: list):
    """
    Get completion from Anthropic API for a multi-turn conversation.

    Args:
        conversation (list): A list of dictionaries, where each dictionary
            represents a turn in the conversation and has 'role' and 'content' keys.

    Returns:
        str: The model's response.
    """
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        messages=conversation # Pass the entire conversation instead of a prompt
    )
    return message.content[0].text


def evaluate_ordinal(model_output, conversation):
    conversation_text = "".join(f"{turn['role']}: {turn['content']}\n" for turn in conversation[:-1])

    ordinal_prompt = f"""Rate how well this response utilizes the conversation context on a scale of 1-5:
    <conversation>
    {conversation_text}
    </conversation>
    <response>{model_output}</response>
    1: Completely ignores context
    5: Perfectly utilizes context
    Output only the number and nothing else."""

    response = client.messages.create(
        model="claude-3-opus-20240229",
        max_tokens=50,
        messages=[{"role": "user", "content": ordinal_prompt}]

        )
    return int(response.content[0].text.strip())

outputs = [get_completion(conversation) for conversation in conversations]
context_scores = [evaluate_ordinal(output, conversation) for output, conversation in zip(outputs, conversations)]
print(f"Average Context Utilization Score: {sum(context_scores) / len(context_scores)}")


ordinal_scale_ratings = context_scores

Average Context Utilization Score: 3.5


#Dashboard

In [31]:
!pip install dash plotly

import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.graph_objs as go
import pandas as pd
import numpy as np


In [48]:
# Initialize the Dash app
app = dash.Dash(__name__)

# Use extracted variables or defaults if missing
sentiment_accuracy = accuracy * 100  # From sentiment analysis code

def get_completion_faq(prompt: str):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2048,  # Adjust as needed
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

# Initialize the dictionary to store all metrics
extracted_variables = {
    'accuracy': [],
    'cosine_similarity_scores': [],
    'rouge_l_scores': [],
    'likert_scores': [],
    'binary_classification': [],
    'ordinal_scale_ratings': []
}

# Cosine Similarity Scores (use extracted or simulate if empty)
cosine_similarity_scores = extracted_variables['cosine_similarity_scores'] if extracted_variables['cosine_similarity_scores'] else [np.random.uniform(70, 100) for _ in range(10)]
rouge_l_scores = extracted_variables['rouge_l_scores']
likert_scores = extracted_variables['likert_scores']
binary_classification = extracted_variables['binary_classification']
ordinal_scale_ratings = extracted_variables['ordinal_scale_ratings']

# DataFrames for Display
cosine_df = pd.DataFrame({"Item": [f"FAQ {i+1}" for i in range(len(cosine_similarity_scores))], "Cosine Similarity": cosine_similarity_scores})
rouge_df = pd.DataFrame({"Item": [f"Text {i+1}" for i in range(len(rouge_l_scores))], "ROUGE-L Score": rouge_l_scores})
likert_df = pd.DataFrame({"Item": [f"Response {i+1}" for i in range(len(likert_scores))], "Likert Score (1-5)": likert_scores})
binary_df = pd.DataFrame({"Item": [f"Eval {i+1}" for i in range(len(binary_classification))], "Classification": binary_classification})
ordinal_df = pd.DataFrame({"Item": [f"Response {i+1}" for i in range(len(ordinal_scale_ratings))], "Ordinal Rating (1-5)": ordinal_scale_ratings})

# Layout of the dashboard
app.layout = html.Div([
    html.H1("NLP Model Evaluation Dashboard", style={'textAlign': 'center'}),

    # Sentiment Analysis Accuracy Gauge
    html.Div([
        dcc.Graph(
            id='sentiment-accuracy-gauge',
            figure=go.Figure(go.Indicator(
                mode="gauge+number",
                value=sentiment_accuracy,
                title={'text': "Sentiment Analysis Accuracy"},
                gauge={'axis': {'range': [0, 100]},
                       'bar': {'color': "darkblue"},
                       'steps': [
                           {'range': [0, 50], 'color': "red"},
                           {'range': [50, 75], 'color': "yellow"},
                           {'range': [75, 100], 'color': "green"}]}
            ))
        )
    ], style={'width': '50%', 'margin': 'auto'}),

    html.H2("Cosine Similarity Evaluation", style={'textAlign': 'center'}),
    dash_table.DataTable(
        id='cosine-table',
        columns=[{"name": i, "id": i} for i in cosine_df.columns],
        data=cosine_df.to_dict('records'),
        style_table={'width': '80%', 'margin': 'auto'},
        style_cell={'textAlign': 'center'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),
    dcc.Graph(
        id='cosine-bar-chart',
        figure={
            'data': [
                go.Bar(
                    x=cosine_df["Item"],
                    y=cosine_df["Cosine Similarity"],
                    marker_color='blue'
                )
            ],
            'layout': go.Layout(
                title='Cosine Similarity Scores',
                xaxis={'title': 'FAQ'},
                yaxis={'title': 'Similarity (%)'},
                margin={'b': 150},
            )
        }
    ),

    html.H2("ROUGE-L Evaluation", style={'textAlign': 'center'}),
    dash_table.DataTable(
        id='rouge-table',
        columns=[{"name": i, "id": i} for i in rouge_df.columns],
        data=rouge_df.to_dict('records'),
        style_table={'width': '80%', 'margin': 'auto'}, # corrected width
        style_cell={'textAlign': 'center'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),
    dcc.Graph(
        id='rouge-bar-chart',
        figure={
            'data': [
                go.Bar(
                    x=rouge_df["Item"],
                    y=rouge_df["ROUGE-L Score"],
                    marker_color='green'
                )
            ],
            'layout': go.Layout(
                title='ROUGE-L Scores',
                xaxis={'title': 'Text'},
                yaxis={'title': 'ROUGE-L Score'},
                margin={'b': 150},
            )
        }
    ),

    html.H2("LLM-based Likert Scale", style={'textAlign': 'center'}),
    dash_table.DataTable(
        id='likert-table',
        columns=[{"name": i, "id": i} for i in likert_df.columns],
        data=likert_df.to_dict('records'),
        style_table={'width': '80%', 'margin': 'auto'},
        style_cell={'textAlign': 'center'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),
    dcc.Graph(
        id='likert-bar-chart',
        figure={
            'data': [
                go.Bar(
                    x=likert_df["Item"],
                    y=likert_df["Likert Score (1-5)"],
                    marker_color='orange'
                )
            ],
            'layout': go.Layout(
                title='Likert Scale Ratings',
                xaxis={'title': 'Response'},
                yaxis={'title': 'Score'},
                margin={'b': 150},
            )
        }
    ),

    html.H2("LLM-based Binary Classification", style={'textAlign': 'center'}),
    dash_table.DataTable(
        id='binary-table',
        columns=[{"name": i, "id": i} for i in binary_df.columns],
        data=binary_df.to_dict('records'),
        style_table={'width': '80%', 'margin': 'auto'},
        style_cell={'textAlign': 'center'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),

    html.H2("LLM-based Ordinal Scale", style={'textAlign': 'center'}),
    dash_table.DataTable(
        id='ordinal-table',
        columns=[{"name": i, "id": i} for i in ordinal_df.columns],
        data=ordinal_df.to_dict('records'),
        style_table={'width': '80%', 'margin': 'auto'},
        style_cell={'textAlign': 'center'},
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'}
    ),
    dcc.Graph(
        id='ordinal-bar-chart',
        figure={
            'data': [
                go.Bar(
                    x=ordinal_df["Item"],
                    y=ordinal_df["Ordinal Rating (1-5)"],
                    marker_color='purple'
                )
            ],
            'layout': go.Layout(
                title='Ordinal Scale Ratings',
                xaxis={'title': 'Response'},
                yaxis={'title': 'Rating'},
                margin={'b': 150},
            )
        }
    ),
])

# Run the app
if __name__ == '__main__':
    app.run_server(debug=False, port=8050)

<IPython.core.display.Javascript object>